In [1]:
import torch

print(torch.__version__)

2.1.2


In [2]:
torch.cuda.is_available()

False

In [3]:
torch.backends.mps.is_available()

True

In [4]:
torch.tensor(1)
torch.tensor([1,2,3])
tensor2d = torch.tensor([[1,2],[3,4]])
tensor2d.shape
tensor3d= torch.tensor([[[1,2],[3,4]],
              [[5,6],[7,9]]])
tensor3d.shape

torch.Size([2, 2, 2])

In [5]:
import numpy as np

array3d = np.array([[[1,2],[3,4]],
              [[5,6],[7,9]]])
tensor1 = torch.tensor(array3d) #copy
tensor2 = torch.from_numpy(array3d) #sharing memory

In [6]:
array3d[0,0,0] = -1
array3d

array([[[-1,  2],
        [ 3,  4]],

       [[ 5,  6],
        [ 7,  9]]])

In [7]:
tensor2

tensor([[[-1,  2],
         [ 3,  4]],

        [[ 5,  6],
         [ 7,  9]]])

In [8]:
tensor2d.reshape(1,4).T


tensor([[1],
        [2],
        [3],
        [4]])

In [9]:
tensor2d @ tensor2d

tensor([[ 7, 10],
        [15, 22]])

In [10]:
import torch.nn.functional as F

### z = x * w + b
y = torch.tensor([1.0])
w = torch.tensor([1.1])
x = torch.tensor([2.2])
b = torch.tensor([0.0])

z= x*w + b
print("z:", z)
a = torch.sigmoid(z)
print("a:", a)
loss = F.binary_cross_entropy(a,y)
print("loss:",loss)
print(-torch.log(a))

z: tensor([2.4200])
a: tensor([0.9183])
loss: tensor(0.0852)
tensor([0.0852])


In [11]:
from torch.autograd import grad

y = torch.tensor([1.0])
w = torch.tensor([1.1], requires_grad=True)
x = torch.tensor([2.2])
b = torch.tensor([0.0], requires_grad=True)

z= x*w + b
print("z:", z)
a = torch.sigmoid(z)
print("a:", a)
loss = F.binary_cross_entropy(a,y)

gradw = grad(loss,w, retain_graph=True)
gradb = grad(loss,b, retain_graph=True)

print(gradw, gradb)

loss.backward()

print(w.grad, b.grad)

z: tensor([2.4200], grad_fn=<AddBackward0>)
a: tensor([0.9183], grad_fn=<SigmoidBackward0>)
(tensor([-0.1797]),) (tensor([-0.0817]),)
tensor([-0.1797]) tensor([-0.0817])


In [12]:
class NeuralNetwork(torch.nn.Module):

    def __init__(self, num_input, num_output):
        super().__init__()

        self.layers = torch.nn.Sequential(
            torch.nn.Linear(num_input, 30),
            torch.nn.ReLU(),
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),
            torch.nn.Linear(20, num_output),
        )

    def forward(self,x):
        return self.layers(x)

In [13]:
model_test = NeuralNetwork(50, 3) 

In [14]:
print(model_test)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


In [15]:
model_test.layers[0].bias.shape

sum([p.numel() for p in model_test.parameters() if p.requires_grad])

2213

In [16]:
X = torch.rand((1,50))
with torch.no_grad():
    print(torch.softmax(model_test(X), dim=1))

tensor([[0.3385, 0.3780, 0.2834]])


In [17]:
from torch.utils.data import Dataset

X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [2.3, -1.1],
    [1.2, 3.1],
    [-1.9, 29],
    [4.3, -1]
])

y_train = torch.tensor([0,0, 1,1,1,0])


X_test = torch.tensor([
    [-1.3, 5.1],
    [0.9, 3.9],
    [3.3, -1.1]
])

y_test = torch.tensor([1,0, 1])


class ToyDataset(Dataset):
    def __init__(self, X,y):
        super().__init__()
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

    def __len__(self):
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

In [18]:
len(train_ds)

6

In [19]:
from torch.utils.data import DataLoader 

train_loader = DataLoader(
    dataset = train_ds,
    batch_size= 2,
    shuffle = True,
    num_workers = 0
)


for idx, (x,y) in enumerate(train_loader):
    print("Batch ",idx, x,y) 

Batch  0 tensor([[-1.2000,  3.1000],
        [ 1.2000,  3.1000]]) tensor([0, 1])
Batch  1 tensor([[-0.9000,  2.9000],
        [ 4.3000, -1.0000]]) tensor([0, 0])
Batch  2 tensor([[-1.9000, 29.0000],
        [ 2.3000, -1.1000]]) tensor([1, 1])


In [20]:
test_loader = DataLoader(
    dataset = test_ds,
    batch_size= 2,
    shuffle = False,
    num_workers = 0
)

In [21]:
model = NeuralNetwork(2,2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
num_epochs = 3 

for epoch in range(num_epochs):
    model.train()
    for idx, (x,y) in enumerate(train_loader):
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print("Epoch", epoch, "Batch", idx, f"loss: {loss:.2f}")



Epoch 0 Batch 0 loss: 0.72
Epoch 0 Batch 1 loss: 1.37
Epoch 0 Batch 2 loss: 0.56
Epoch 1 Batch 0 loss: 1.25
Epoch 1 Batch 1 loss: 0.75
Epoch 1 Batch 2 loss: 0.82
Epoch 2 Batch 0 loss: 8.20
Epoch 2 Batch 1 loss: 1.20
Epoch 2 Batch 2 loss: 0.69


In [38]:
from importlib.metadata import version

print("tiktiken version", version("tiktoken"))

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

raw_text


tiktiken version 0.6.0


'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)\n\n"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it\'s going to send the value of my picture \'way up; but I don\'t think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing\'s lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn\'s "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?\n\nWell!--even 

In [23]:
import re

preprocessed = [item.strip() for item in re.split(r'(\s|--|[()\'"!_?:.,])', raw_text) if item.strip()]
all_words = sorted(set(preprocessed))
all_words
vocab_size = len(all_words)

In [24]:
vocab = {token:i for i, token in enumerate(all_words)}
vocab

{'!': 0,
 '"': 1,
 "'": 2,
 '(': 3,
 ')': 4,
 ',': 5,
 '--': 6,
 '.': 7,
 ':': 8,
 ';': 9,
 '?': 10,
 'A': 11,
 'Ah': 12,
 'Among': 13,
 'And': 14,
 'Are': 15,
 'Arrt': 16,
 'As': 17,
 'At': 18,
 'Be': 19,
 'Begin': 20,
 'Burlington': 21,
 'But': 22,
 'By': 23,
 'Carlo': 24,
 'Carlo;': 25,
 'Chicago': 26,
 'Claude': 27,
 'Come': 28,
 'Croft': 29,
 'Destroyed': 30,
 'Devonshire': 31,
 'Don': 32,
 'Dubarry': 33,
 'Emperors': 34,
 'Florence': 35,
 'For': 36,
 'Gallery': 37,
 'Gideon': 38,
 'Gisburn': 39,
 'Gisburns': 40,
 'Grafton': 41,
 'Greek': 42,
 'Grindle': 43,
 'Grindles': 44,
 'HAD': 45,
 'Had': 46,
 'Hang': 47,
 'Has': 48,
 'He': 49,
 'Her': 50,
 'Hermia': 51,
 'His': 52,
 'How': 53,
 'I': 54,
 'If': 55,
 'In': 56,
 'It': 57,
 'Jack': 58,
 'Jove': 59,
 'Just': 60,
 'Lord': 61,
 'Made': 62,
 'Miss': 63,
 'Money': 64,
 'Monte': 65,
 'Moon-dancers': 66,
 'Mr': 67,
 'Mrs': 68,
 'My': 69,
 'Never': 70,
 'No': 71,
 'Now': 72,
 'Nutley': 73,
 'Of': 74,
 'Oh': 75,
 'On': 76,
 'Once': 77,


In [25]:
class TokenizerV1: 
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}

    def encode(self,text):
        processed_text = [item.strip() for item in re.split(r'(\s|--|[()\'"!_?:.,])', text) if item.strip()]
        return [self.str_to_int[word] for word in processed_text]

    def decode(self, ids):
        return  " ".join(self.int_to_str[id] for id in ids)

t1 = TokenizerV1(vocab)
t1.decode(t1.encode(raw_text[:33]))

#t1.encode("It's the last he painted, you you test")

'I HAD always thought Jack Gisburn'

In [26]:
all_words.extend(["<|unk|>"])
vocab = {token:i for i, token in enumerate(all_words)}

In [28]:



class TokenizerV2: 
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}

    def encode(self,text):
        processed_text = [item.strip() for item in re.split(r'(\s|--|[()\'"!_?:.,])', text) if item.strip()]
        processed_text = [item if item in self.str_to_int else "<|unk|>" for item in processed_text]
        return [self.str_to_int[word] for word in processed_text]

    def decode(self, ids):
        return  " ".join(self.int_to_str[id] for id in ids)

t2 = TokenizerV2(vocab)
t2.decode(t2.encode("It's the last he painted, you you test"))

"It ' s the last he painted , you you <|unk|>"

## BPE

In [30]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

[(tokenizer.decode([id]), id) for id in range(30)]
tokenizer.max_token_value

50256

In [35]:
text = "Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace."

tokenizer.decode(tokenizer.encode(text, allowed_special={"<|endoftext|>"}))


enc_text = tokenizer.encode(raw_text)
enc_text

[40,
 367,
 2885,
 1464,
 1807,
 3619,
 402,
 271,
 10899,
 2138,
 257,
 7026,
 15632,
 438,
 2016,
 257,
 922,
 5891,
 1576,
 438,
 568,
 340,
 373,
 645,
 1049,
 5975,
 284,
 502,
 284,
 3285,
 326,
 11,
 287,
 262,
 6001,
 286,
 465,
 13476,
 11,
 339,
 550,
 5710,
 465,
 12036,
 11,
 6405,
 257,
 5527,
 27075,
 11,
 290,
 4920,
 2241,
 287,
 257,
 4489,
 64,
 319,
 262,
 34686,
 41976,
 13,
 357,
 10915,
 314,
 2138,
 1807,
 340,
 561,
 423,
 587,
 10598,
 393,
 28537,
 2014,
 198,
 198,
 1,
 464,
 6001,
 286,
 465,
 13476,
 1,
 438,
 5562,
 373,
 644,
 262,
 1466,
 1444,
 340,
 13,
 314,
 460,
 3285,
 9074,
 13,
 46606,
 536,
 5469,
 438,
 14363,
 938,
 4842,
 1650,
 353,
 438,
 2934,
 489,
 3255,
 465,
 48422,
 540,
 450,
 67,
 3299,
 13,
 366,
 5189,
 1781,
 340,
 338,
 1016,
 284,
 3758,
 262,
 1988,
 286,
 616,
 4286,
 705,
 1014,
 510,
 26,
 475,
 314,
 836,
 470,
 892,
 286,
 326,
 11,
 1770,
 13,
 8759,
 2763,
 438,
 1169,
 2994,
 284,
 943,
 17034,
 318,
 477,
 314,
 892,


In [39]:
class GPTDatasetV1(Dataset):

    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        for i in range(0, len(token_ids)-max_length, stride):
            input_chunk = token_ids[i: i+max_length]
            target_chunk = token_ids[i+1: i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.target_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

def create_dataloader_v1(txt, batch_size = 4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers = 0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    return DataLoader(
        dataset, 
        batch_size = batch_size,
        shuffle=shuffle,
        drop_last = drop_last,
        num_workers = num_workers
    )


In [43]:
dataloader = create_dataloader_v1(raw_text,
                                  batch_size = 8,
                                  max_length=4,
                                  stride=4, shuffle=False)

data_iter = iter(dataloader)
next(data_iter)

[tensor([[   40,   367,  2885,  1464],
         [ 1807,  3619,   402,   271],
         [10899,  2138,   257,  7026],
         [15632,   438,  2016,   257],
         [  922,  5891,  1576,   438],
         [  568,   340,   373,   645],
         [ 1049,  5975,   284,   502],
         [  284,  3285,   326,    11]]),
 tensor([[  367,  2885,  1464,  1807],
         [ 3619,   402,   271, 10899],
         [ 2138,   257,  7026, 15632],
         [  438,  2016,   257,   922],
         [ 5891,  1576,   438,   568],
         [  340,   373,   645,  1049],
         [ 5975,   284,   502,   284],
         [ 3285,   326,    11,   287]])]